# 02 — Extraction promptbook: development runs

**Purpose.** Iterate the stance-extraction prompt on a small development
sample until its outputs read correctly, *before* any gold coding or
production run. This is the only notebook where the prompt is allowed to
change; the freeze that ends development renames `PROMPT_VERSION` in
`lib/stance.py` from `-dev` to `1.0`, and from that point the wording is
immutable (rule V1: never tune on data used to report accuracy).

**What is being extracted.** Per legislator turn, stance records
`(sentence numbers, neutral-noun-phrase topic, support | concern)` — the
atomic unit of Track 2 in the Stage-3 plan. Sentences were numbered
deterministically in `01`; the model selects numbers, so evidence cannot be
hallucinated, and `stance.parse_records` machine-checks every output (JSON
shape, stance vocabulary, index validity, topic directionality lint).

**What this notebook does NOT do.** No accuracy claims — there are no gold
labels yet (`03`). No model race — that comparison (Qwen3.5-4B vs
Llama-3.1-8B, decided on dev-gold accuracy per rule V3) happens once the dev
wording stabilizes. No production — `04` and its sbatch. The deliverable here
is a read: dev-sample outputs printed next to their source sentences, plus
the machine-check rates.

**Inputs.** `interim/topics/track2/member_turns.parquet` — from `01`.

**Outputs.** Call logs under `interim/topics/dev/status/` (every dev call is
logged with the prompt version that produced it, so dev iterations stay
distinguishable). Nothing else persisted — development output is for reading.

## Setup

Weights on scratch, token from the gitignored `.env`, generation parameters
reused from Stage 2 (`summarize.GENERATION`: temperature 0.1, do_sample) with
the same per-document seeding, so a re-run of this notebook reproduces its
outputs bit for bit.

In [1]:
import hashlib
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, "../lib")

from paths import HF_CACHE, topics_run_dir

HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
for line in (Path(".env").read_text().splitlines() if Path(".env").exists() else []):
    if line.startswith("HF_TOKEN="):
        os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip()

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import stance
import summarize as S

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
# 1024: a long window can legitimately carry many records, and the 0.1-dev run
# showed 320 truncating mid-JSON on long opening statements. Generation stops
# at EOS, so the larger cap costs nothing on short turns.
MAX_NEW_TOKENS = 1024
N_DEV = 30

TRACK2 = topics_run_dir("track2")
DEV_DIR = topics_run_dir("dev")

print(f"model    : {MODEL}")
print(f"prompt   : {stance.PROMPT_VERSION}")
print(f"window   : {stance.WINDOW_SENTENCES} sentences")
print(f"sampling : {S.GENERATION}")
print(f"gpu      : {torch.cuda.get_device_name(0)}")

model    : Qwen/Qwen3.5-4B
prompt   : stance-extraction-0.3-dev
window   : 40 sentences
sampling : {'do_sample': True, 'temperature': 0.1}
gpu      : NVIDIA A40


## The prompt, in full

Printed, not described — the record discipline from Stage 2. A prompt that is
not in the notebook output is not a reported prompt.

In [2]:
print("=" * 78)
print(f"SYSTEM   ({stance.PROMPT_VERSION})")
print("=" * 78)
print(stance.SYSTEM)
print()
print("=" * 78)
print("USER")
print("=" * 78)
print(stance.USER)

SYSTEM   (stance-extraction-0.3-dev)
You are an expert reader of United States congressional hearing transcripts. You report only what the text states.

USER
Below is one speaking turn by a member of Congress, split into numbered sentences.

Task: find every sentence in which the member takes a directional position on a policy object, and return one record per position.

A record has three fields:
- "sentences": the sentence number(s) expressing the position (1-3 consecutive numbers);
- "topic": the policy object, as a short and specific neutral noun phrase (the policy, program, regulation, technology, practice, or budget item itself; never directional wording; name the specific object, not the agency or field around it);
- "stance": "support" if the member endorses, advocates, or urges adoption, funding, or expansion of the topic; "concern" if the member expresses worry, criticism, or objection toward the topic, or urges its restriction, delay, or scrutiny.

The stance is toward the t

## The development sample

Thirty turns, stratified over chamber and turn length. Length matters because
the failure modes differ: long opening statements risk missed records
(recall), short procedural turns risk invented ones (precision) — both must
be visible during development. Short procedural turns are *deliberately*
included: the correct output for most of them is an empty list, and a prompt
that cannot return empty lists fails the design. Seeded, so every dev
iteration reads the same thirty turns and differences between iterations are
attributable to the wording, not the sample.

In [3]:
turns = pd.read_parquet(TRACK2 / "member_turns.parquet")
turns = turns.loc[turns["n_sentences"] > 0]

turns["bucket"] = pd.cut(turns["n_sentences"], [0, 3, 12, 40, 10_000],
                          labels=["short", "medium", "long", "very_long"])
per_cell = max(1, N_DEV // (2 * 4))
dev = (turns.groupby(["chamber", "bucket"], observed=True, group_keys=False)
       .apply(lambda g: g.sample(min(per_cell, len(g)), random_state=1089))
       .head(N_DEV).reset_index(drop=True))

print(dev.groupby(["chamber", "bucket"], observed=True).size().to_string())
print(f"\n{len(dev)} dev turns, {dev['n_sentences'].sum():,} sentences")

chamber  bucket   
HOUSE    short        3
         medium       3
         long         3
         very_long    3
JOINT    short        3
         medium       3
         long         3
         very_long    2
SENATE   short        3
         medium       3
         long         1

30 dev turns, 591 sentences


/tmp/SLURM_11256507/ipykernel_1458895/873130140.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(per_cell, len(g)), random_state=1089))


## Load the model

In [4]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s — "
      f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

loaded in 5s — 4.21B params


## Run extraction on the dev sample

One turn per call, fresh context (rule P3). Each call is seeded from the turn
key and logged through the Stage-2 `CallLog` — stage name `extract`, with the
full output preserved, so a later reader can reconstruct any dev iteration
from disk.

In [5]:
results = []
started = time.time()
n_calls = 0
for row in dev.itertuples():
    key = f"{row.packageId}:{row.turn_idx}"
    torch.manual_seed(int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big"))

    speaker = f"{row.honorific or ''} {row.surname or ''}".strip() or "Unidentified"
    log = S.CallLog(DEV_DIR / "status" / f"{row.packageId}_{row.turn_idx}.jsonl",
                    key, MODEL)

    # Long turns are extracted in windows with turn-global sentence numbering,
    # so a record's indices mean the same thing whichever window produced them.
    records, errors = [], []
    for w_index, (w_start, w_sents) in enumerate(stance.windows(list(row.sentences))):
        user = stance.render_user(w_sents, speaker, start=w_start)
        t0 = time.time()
        output, hit_cap = S._generate(model, tokenizer, stance.SYSTEM, user,
                                       max_new_tokens=MAX_NEW_TOKENS)
        log.write("extract", 0, w_index, 1, user, output, tokenizer,
                  time.time() - t0)
        n_calls += 1
        w_records, w_errors = stance.parse_records(output, len(w_sents),
                                                    start=w_start)
        if hit_cap:
            w_errors.append("hit_token_cap")
        records.extend(w_records)
        errors.extend(w_errors)

    # Deterministic insurance: one position split across adjacent sentences
    # collapses to one record, so intensity counts do not reward verbosity.
    records, n_merged = stance.merge_adjacent_records(records)

    results.append({"key": key, "speaker": speaker, "sentences": row.sentences,
                    "records": records, "errors": errors, "n_merged": n_merged})

total_merged = sum(r["n_merged"] for r in results)
print(f"{len(results)} turns / {n_calls} calls in {time.time() - started:.0f}s "
      f"({(time.time() - started) / n_calls:.1f}s per call); "
      f"{total_merged} adjacent duplicates merged")

30 turns / 39 calls in 86s (2.2s per call); 1 adjacent duplicates merged


## Read the output

The point of a dev notebook. Every turn is printed with its numbered
sentences and the records extracted from them, so wording problems are found
by reading, not inferred from rates. Things to look for, from the plan's
worked examples: questions-without-position wrongly extracted, directional
topic phrases (the lint marks them), missed concern-phrased positions, and
empty lists on turns that plainly take a position.

In [6]:
for r in results:
    print("=" * 78)
    print(f"{r['key']}  [{r['speaker']}]  "
          f"{len(r['records'])} record(s)"
          + (f"  ERRORS: {r['errors']}" if r["errors"] else ""))
    print(stance.format_numbered(list(r["sentences"])[:12]))
    if len(r["sentences"]) > 12:
        print(f"  ... {len(r['sentences']) - 12} more sentences")
    for rec in r["records"]:
        lint = f"  [LINT: {rec['topic_lint']}]" if rec["topic_lint"] else ""
        print(f"  -> {rec['sentences']}  ({rec['stance']:<7})  "
              f"{rec['topic']}{lint}")
    if not r["records"] and not r["errors"]:
        print("  -> (no directional position — empty list)")
    print()

CHRG-109hhrg20007:162  [Mr. Udall]  0 record(s)
[1] Dr. Benzakein.
  -> (no directional position — empty list)

CHRG-114hhrg97769:74  [Ms. Esty]  0 record(s)
[1] And if you can briefly talk a little bit about, for example, the Home Energy Solutions program, how that works in Connecticut again to put the power in the hands of consumers and businesses to make those choices.
  -> (no directional position — empty list)

CHRG-111hhrg57174:188  [Mr. Garamendi]  0 record(s)
[1] And you don't think you will have to pay for that?
  -> (no directional position — empty list)

CHRG-115hhrg28414:167  [Mr. Dunn]  0 record(s)
[1] Thank you.
[2] So Mr. Gerstenmaier, in the cost end of the training side, you now have three new, different manned capsules.
[3] And I presume your astronauts have to train on each of these.
[4] But going forward, how are you going to segment that training?
[5] Is it everybody trains on everything or 1/3 here, 1/3 there or how does that work?
  -> (no directional position — 

## Machine-check summary

The rates that later become per-shard alarm bands in production. On thirty
turns they are direction-finding only — but a parse rate visibly below one,
or lint hits on a large share of topics, is a wording problem to fix now,
while changing the prompt is still legal.

In [7]:
from collections import Counter

n_records = sum(len(r["records"]) for r in results)
error_counts = Counter(e for r in results for e in r["errors"])
stances = Counter(rec["stance"] for r in results for rec in r["records"])
lints = [rec["topic"] for r in results for rec in r["records"] if rec["topic_lint"]]

clean = sum(1 for r in results if not r["errors"])
print(f"turns parsed clean : {clean}/{len(results)}")
print(f"records extracted  : {n_records} "
      f"({n_records / len(results):.1f} per turn)")
print(f"empty-list turns   : {sum(1 for r in results if not r['records'])}")
print(f"stance mix         : {dict(stances)}")
print(f"errors             : {dict(error_counts) or 'none'}")
print(f"topic lint hits    : {len(lints)}")
for topic in lints[:5]:
    print(f"   {topic!r}")

turns parsed clean : 29/30
records extracted  : 38 (1.3 per turn)
empty-list turns   : 24
stance mix         : {'support': 12, 'concern': 26}
errors             : {'indices_not_consecutive': 1}
topic lint hits    : 2
   'NASA Inspector General Robert Cobb blocking notification to State Department'
   'NASA Inspector General Robert Cobb blocking NASA OIG investigation'


## What happens next (and what must not happen here)

1. **Iterate.** Read the output above with the researcher; adjust wording in
   `lib/stance.py`; re-run this notebook. Every iteration's calls are on disk
   under `interim/topics/dev/status/`, tagged with the prompt version.
2. **Race the models** on the dev sample once wording stabilizes
   (Qwen3.5-4B vs Llama-3.1-8B-Instruct) — the production model is chosen on
   measured behavior *before* the final polish, so the prompt is not silently
   tuned to one model (rule V3).
3. **Freeze.** Bump `stance.PROMPT_VERSION` to `stance-extraction-1.0`,
   record the file hash. After this point the wording never changes.
4. **Gold-code** (notebook `03`): ~300 turns, two human coders, the same
   instruction text — only then do accuracy numbers exist.

The one forbidden move: looking at validation or test turns while iterating
here. The dev sample is these thirty turns; everything else stays unseen.